# Options Data Analysis from Account Overview

This notebook processes the options data from the Schwab account overview CSV file and creates a structured pandas DataFrame for analysis. The notebook follows modular design principles with comprehensive type hints and detailed commenting.

In [11]:
# Import Required Libraries
import pandas as pd  # Data manipulation and analysis library
import numpy as np   # Numerical computing library
import os           # Operating system interface for environment variables
import re           # Regular expressions for text processing
from typing import List, Dict, Optional, Tuple, Any  # Type hints for better code documentation
import logging      # Logging functionality for debugging

# Configure logging for debugging purposes
logging.basicConfig(level=logging.DEBUG if os.getenv("DEBUG") == "1" else logging.INFO)
logger = logging.getLogger(__name__)  # Create logger instance for this module

logger.info("Successfully imported all required libraries for options data analysis")

INFO:__main__:Successfully imported all required libraries for options data analysis


In [12]:
# Read and Parse the CSV File
def read_csv_file(file_path: str) -> List[str]:
    """
    Read the entire CSV file and return as a list of lines.
    
    Args:
        file_path (str): Path to the CSV file to be read
        
    Returns:
        List[str]: List of strings, each representing a line from the file
        
    Raises:
        FileNotFoundError: If the specified file does not exist
        IOError: If there's an error reading the file
    """
    logger.debug(f"Attempting to read file: {file_path}")
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:  # Open file with UTF-8 encoding
            lines = file.readlines()  # Read all lines into memory
        
        logger.info(f"Successfully read {len(lines)} lines from {file_path}")
        return lines
    
    except FileNotFoundError as e:
        logger.error(f"File not found: {file_path}")
        raise e
    except IOError as e:
        logger.error(f"Error reading file {file_path}: {str(e)}")
        raise e

# Define the file path for the overview CSV
csv_file_path: str = "/home/hai/hai_vscode/MyDevelopment/data/overview.csv"
logger.debug(f"Using CSV file path: {csv_file_path}")

# Read the CSV file content
file_lines: List[str] = read_csv_file(csv_file_path)
logger.info(f"File reading completed. Total lines: {len(file_lines)}")

INFO:__main__:Successfully read 159 lines from /home/hai/hai_vscode/MyDevelopment/data/overview.csv
INFO:__main__:File reading completed. Total lines: 159


In [13]:
# Extract Options Data Section
def find_options_section(lines: List[str]) -> Tuple[int, int]:
    """
    Locate the Options section in the CSV file lines.
    
    Args:
        lines (List[str]): List of lines from the CSV file
        
    Returns:
        Tuple[int, int]: Start and end line indices for the Options section
        
    Raises:
        ValueError: If Options section is not found or malformed
    """
    logger.debug("Searching for Options section in file lines")
    
    options_start: int = -1  # Initialize start index as not found
    options_end: int = -1    # Initialize end index as not found
    
    # Iterate through all lines to find the Options section
    for i, line in enumerate(lines):
        stripped_line: str = line.strip()  # Remove leading/trailing whitespace
        
        # Check if this line indicates the start of Options section
        if stripped_line == "Options":
            options_start = i
            logger.debug(f"Found Options section start at line {i}")
            continue
        
        # If we're past the Options start, look for the end
        if options_start != -1 and stripped_line == "Others":
            options_end = i
            logger.debug(f"Found Options section end at line {i}")
            break
    
    # Validate that we found both start and end
    if options_start == -1:
        raise ValueError("Options section not found in the CSV file")
    
    if options_end == -1:
        # If no explicit end found, use end of file
        options_end = len(lines)
        logger.debug(f"No explicit end found, using end of file at line {options_end}")
    
    logger.info(f"Options section found from line {options_start} to line {options_end}")
    return options_start, options_end

def extract_options_lines(lines: List[str], start_idx: int, end_idx: int) -> List[str]:
    """
    Extract the options data lines from the specified range.
    
    Args:
        lines (List[str]): All lines from the CSV file
        start_idx (int): Starting line index for options section
        end_idx (int): Ending line index for options section
        
    Returns:
        List[str]: Lines containing options data only
    """
    logger.debug(f"Extracting options lines from index {start_idx} to {end_idx}")
    
    # Extract the section and filter out empty lines and totals
    options_lines: List[str] = []
    
    for i in range(start_idx + 1, end_idx):  # Skip the "Options" header line
        line: str = lines[i].strip()  # Remove whitespace
        
        # Skip empty lines and overall totals line
        if line and not line.startswith(",,,,,,") and "OVERALL TOTALS" not in line:
            options_lines.append(line)
    
    logger.info(f"Extracted {len(options_lines)} options data lines")
    return options_lines

# Find and extract the options section
options_start_idx, options_end_idx = find_options_section(file_lines)
raw_options_lines: List[str] = extract_options_lines(file_lines, options_start_idx, options_end_idx)

# Display first few lines for verification
logger.debug("First 3 options lines:")
for i, line in enumerate(raw_options_lines[:3]):
    logger.debug(f"Line {i}: {line}")

INFO:__main__:Options section found from line 75 to line 88
INFO:__main__:Extracted 10 options data lines


In [14]:
# Clean and Structure Options Data
def clean_monetary_value(value: str) -> float:
    """
    Clean and convert monetary string values to float.
    
    Args:
        value (str): Raw monetary string value (may contain $, commas, parentheses)
        
    Returns:
        float: Cleaned numerical value (negative if in parentheses)
    """
    if not value or value.strip() == "":  # Handle empty values
        return 0.0
    
    cleaned_value: str = value.strip()  # Remove leading/trailing whitespace
    
    # Check if value is negative (enclosed in parentheses)
    is_negative: bool = cleaned_value.startswith('(') and cleaned_value.endswith(')')
    
    if is_negative:
        cleaned_value = cleaned_value[1:-1]  # Remove parentheses
    
    # Remove dollar signs, commas, and other formatting characters
    cleaned_value = re.sub(r'[\$,]', '', cleaned_value)
    
    try:
        numeric_value: float = float(cleaned_value)
        return -numeric_value if is_negative else numeric_value
    except ValueError:
        logger.warning(f"Could not convert '{value}' to float, returning 0.0")
        return 0.0

def clean_percentage_value(value: str) -> float:
    """
    Clean and convert percentage string values to float.
    
    Args:
        value (str): Raw percentage string value (may contain % and +/- signs)
        
    Returns:
        float: Cleaned percentage as decimal (e.g., "5%" becomes 5.0)
    """
    if not value or value.strip() == "":  # Handle empty values
        return 0.0
    
    cleaned_value: str = value.strip()  # Remove whitespace
    
    # Remove percentage sign and plus sign
    cleaned_value = cleaned_value.replace('%', '').replace('+', '')
    
    try:
        return float(cleaned_value)
    except ValueError:
        logger.warning(f"Could not convert '{value}' to percentage, returning 0.0")
        return 0.0

def parse_options_line(line: str) -> Dict[str, Any]:
    """
    Parse a single options data line into a structured dictionary.
    
    Args:
        line (str): Raw CSV line containing options data
        
    Returns:
        Dict[str, Any]: Structured dictionary with options data fields
    """
    # Split the line by commas, handling quoted values
    parts: List[str] = [part.strip().strip('"') for part in line.split(',')]
    
    # Ensure we have enough parts for a valid options line
    if len(parts) < 12:
        logger.warning(f"Incomplete options line with {len(parts)} parts: {line}")
        return {}
    
    # Map the parts to structured data according to the CSV header
    options_data: Dict[str, Any] = {
        'symbol': parts[0],                                    # Stock symbol
        'expiration': parts[1],                               # Expiration date
        'strike': float(parts[2]) if parts[2] else 0.0,      # Strike price
        'option_type': parts[3],                              # PUT or CALL
        'quantity': int(parts[4]) if parts[4] else 0,         # Number of contracts
        'trade_price': float(parts[5]) if parts[5] else 0.0,  # Original trade price
        'mark': float(parts[6]) if parts[6] else 0.0,         # Current mark price
        'mark_value': clean_monetary_value(parts[7]),         # Current mark value
        'pl_open': clean_monetary_value(parts[8]),            # P&L from open
        'pl_day': clean_monetary_value(parts[9]),             # Daily P&L
        'pl_percent': clean_percentage_value(parts[10]),      # P&L percentage
        'account_name': parts[11] if len(parts) > 11 else '', # Account name
        'option_code': parts[12] if len(parts) > 12 else ''   # Option code
    }
    
    logger.debug(f"Parsed options data for {options_data['symbol']}")
    return options_data

# Process all options lines
logger.info("Starting to process options lines...")
processed_options: List[Dict[str, Any]] = []

# Skip the header line (first line should be the column headers)
data_lines: List[str] = raw_options_lines[1:] if raw_options_lines else []

for line_num, line in enumerate(data_lines):
    logger.debug(f"Processing options line {line_num + 1}: {line[:50]}...")  # Log first 50 chars
    
    parsed_data: Dict[str, Any] = parse_options_line(line)
    
    if parsed_data:  # Only add if parsing was successful
        processed_options.append(parsed_data)

logger.info(f"Successfully processed {len(processed_options)} options records")

INFO:__main__:Starting to process options lines...
INFO:__main__:Successfully processed 9 options records


In [15]:
# Create Options DataFrame
def create_options_dataframe(options_data: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Create a structured pandas DataFrame from processed options data.
    
    Args:
        options_data (List[Dict[str, Any]]): List of processed options dictionaries
        
    Returns:
        pd.DataFrame: Structured DataFrame with proper data types and indexing
        
    Raises:
        ValueError: If options_data is empty or malformed
    """
    if not options_data:
        raise ValueError("No options data provided to create DataFrame")
    
    logger.debug(f"Creating DataFrame from {len(options_data)} options records")
    
    # Create the DataFrame from the list of dictionaries
    df: pd.DataFrame = pd.DataFrame(options_data)
    
    # Set proper data types for numerical columns
    numeric_columns: List[str] = [
        'strike', 'quantity', 'trade_price', 'mark', 
        'mark_value', 'pl_open', 'pl_day', 'pl_percent'
    ]
    
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')  # Convert to numeric, NaN for errors
    
    # Convert quantity to integer (contracts are whole numbers)
    if 'quantity' in df.columns:
        df['quantity'] = df['quantity'].astype('Int64')  # Use nullable integer type
    
    # Add calculated columns for better analysis
    df['total_contracts'] = abs(df['quantity'])  # Absolute value for total contracts
    df['position_type'] = df['quantity'].apply(lambda x: 'Short' if x < 0 else 'Long')
    df['days_to_expiration'] = None  # Placeholder for future calculation
    
    # Sort by symbol and expiration for better organization
    df = df.sort_values(['symbol', 'expiration'], ascending=[True, True])
    
    # Reset index after sorting
    df = df.reset_index(drop=True)
    
    logger.info(f"Created options DataFrame with shape {df.shape}")
    logger.debug(f"DataFrame columns: {list(df.columns)}")
    
    return df

# Create the options DataFrame
try:
    options_df: pd.DataFrame = create_options_dataframe(processed_options)
    logger.info("Options DataFrame created successfully")
    
    # Display basic information about the DataFrame
    print("Options DataFrame Info:")
    print(f"Shape: {options_df.shape}")
    print(f"Columns: {list(options_df.columns)}")
    print("\nData types:")
    print(options_df.dtypes)
    
except ValueError as e:
    logger.error(f"Error creating DataFrame: {str(e)}")
    options_df = pd.DataFrame()  # Create empty DataFrame as fallback

INFO:__main__:Created options DataFrame with shape (9, 16)
INFO:__main__:Options DataFrame created successfully


Options DataFrame Info:
Shape: (9, 16)
Columns: ['symbol', 'expiration', 'strike', 'option_type', 'quantity', 'trade_price', 'mark', 'mark_value', 'pl_open', 'pl_day', 'pl_percent', 'account_name', 'option_code', 'total_contracts', 'position_type', 'days_to_expiration']

Data types:
symbol                 object
expiration             object
strike                float64
option_type            object
quantity                Int64
trade_price           float64
mark                  float64
mark_value            float64
pl_open               float64
pl_day                float64
pl_percent            float64
account_name           object
option_code            object
total_contracts         Int64
position_type          object
days_to_expiration     object
dtype: object


In [17]:
# Display and Analyze Options Data
def display_options_summary(df: pd.DataFrame) -> None:
    """
    Display comprehensive summary and analysis of options data.
    
    Args:
        df (pd.DataFrame): Options DataFrame to analyze
    """
    if df.empty:
        print("No options data available for analysis")
        return
    
    logger.info("Generating options data summary and analysis")
    
    # Display first few rows
    print("=== OPTIONS POSITIONS OVERVIEW ===")
    print("\nFirst 5 options positions:")
    print(df.to_string(index=False))
    
    # Basic statistics
    print("\n=== BASIC STATISTICS ===")
    print(f"Total number of options positions: {len(df)}")
    print(f"Number of unique symbols: {df['symbol'].nunique()}")
    print(f"Number of long positions: {len(df[df['position_type'] == 'Long'])}")
    print(f"Number of short positions: {len(df[df['position_type'] == 'Short'])}")
    
    # Profit and Loss Analysis
    print("\n=== PROFIT & LOSS ANALYSIS ===")
    total_pl_open: float = df['pl_open'].sum()
    total_pl_day: float = df['pl_day'].sum()
    total_mark_value: float = df['mark_value'].sum()
    
    print(f"Total P&L from Open: ${total_pl_open:,.2f}")
    print(f"Total Daily P&L: ${total_pl_day:,.2f}")
    print(f"Total Mark Value: ${total_mark_value:,.2f}")
    
    # Position breakdown by option type
    print("\n=== POSITION BREAKDOWN BY TYPE ===")
    type_summary = df.groupby(['option_type', 'position_type']).agg({
        'quantity': 'sum',
        'mark_value': 'sum',
        'pl_open': 'sum'
    }).round(2)
    print(type_summary)
    
    # Symbol-level analysis
    print("\n=== SYMBOL-LEVEL ANALYSIS ===")
    symbol_summary = df.groupby('symbol').agg({
        'quantity': 'sum',
        'mark_value': 'sum',
        'pl_open': 'sum',
        'pl_day': 'sum'
    }).round(2)
    symbol_summary = symbol_summary.sort_values('pl_open', ascending=False)
    print(symbol_summary)
    
    # Winners and losers
    print("\n=== TOP PERFORMERS ===")
    winners = df[df['pl_open'] > 0].sort_values('pl_open', ascending=False).head(3)
    losers = df[df['pl_open'] < 0].sort_values('pl_open', ascending=True).head(3)
    
    if not winners.empty:
        print("\nTop 3 Winning Positions:")
        for _, row in winners.iterrows():
            print(f"  {row['symbol']} {row['expiration']} {row['strike']}{row['option_type'][0]}: "
                  f"${row['pl_open']:.2f} ({row['pl_percent']:.1f}%)")
    
    if not losers.empty:
        print("\nTop 3 Losing Positions:")
        for _, row in losers.iterrows():
            print(f"  {row['symbol']} {row['expiration']} {row['strike']}{row['option_type'][0]}: "
                  f"${row['pl_open']:.2f} ({row['pl_percent']:.1f}%)")

def export_options_data(df: pd.DataFrame, output_path: str = "/home/hai/hai_vscode/MyDevelopment/data/processed_options.csv") -> None:
    """
    Export the processed options DataFrame to a CSV file.
    
    Args:
        df (pd.DataFrame): Options DataFrame to export
        output_path (str): Path where to save the processed data
    """
    try:
        df.to_csv(output_path, index=False)
        logger.info(f"Options data exported to {output_path}")
        print(f"\nProcessed options data exported to: {output_path}")
    except Exception as e:
        logger.error(f"Error exporting data: {str(e)}")
        print(f"Error exporting data: {str(e)}")

# Display the comprehensive analysis
display_options_summary(options_df)

# Export the processed data
export_options_data(options_df)

# Display final success message
print("\n" + "="*50)
print("OPTIONS DATA ANALYSIS COMPLETED SUCCESSFULLY")
print("="*50)
logger.info("Options data analysis notebook execution completed successfully")

INFO:__main__:Generating options data summary and analysis


INFO:__main__:Options data exported to /home/hai/hai_vscode/MyDevelopment/data/processed_options.csv
INFO:__main__:Options data analysis notebook execution completed successfully


=== OPTIONS POSITIONS OVERVIEW ===

First 5 options positions:
symbol expiration  strike option_type  quantity  trade_price  mark  mark_value  pl_open  pl_day  pl_percent  account_name    option_code  total_contracts position_type days_to_expiration
   AMD  26 SEP 25   165.0         PUT        -1         4.58 5.400      -540.0   -82.66  127.65      -18.07       HaiRoth  AMD250926P165                1         Short               None
  BMNR  10 OCT 25    50.0         PUT        -1         2.46 2.480      -248.0    -2.00   -2.00       -0.81        HaiIRA  BMNR251010P50                1         Short               None
  BMNR  17 OCT 25    50.0         PUT        -1         3.46 3.375      -337.5     8.50    8.50        2.46 HaiIndividual  BMNR251017P50                1         Short               None
  BMNR  24 OCT 25    48.0         PUT        -1         3.45 3.325      -332.5    12.50   12.50        3.62        HaiIRA  BMNR251024P48                1         Short               None
  